# 01 — Limpieza y Validación de Datos

Carga `zonas_ageb_clean_v3.json` (2 453 AGEBs: 2 431 urbanas + 22 rurales),
valida integridad y genera el dataset procesado para modelado.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from src.config import setup_logging, ensure_output_dirs, FEATURE_COLS_RAW, TARGET_COL
from src.data_loader import load_ageb_v3, EXPECTED_ROWS
from src.preprocessing import create_target_variable, engineer_features
from src.utils import save_path, validate_dataframe

setup_logging()
ensure_output_dirs()

## 1. Carga de datos (fuente única: v3)

In [ ]:
df = load_ageb_v3()
print(f'Registros cargados: {len(df)} (esperados: {EXPECTED_ROWS})')
assert len(df) == EXPECTED_ROWS
df.head()

## 2. Auditoría de calidad

In [ ]:
print('Shape:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)
print('\nNulos por columna:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nDuplicados CVEGEO:', df.duplicated(subset='CVEGEO').sum())

In [ ]:
# Estadísticas descriptivas de features físicas
df[FEATURE_COLS_RAW].describe().T

## 3. Creación del target binarizado

In [ ]:
df = create_target_variable(df)
print(df['alto_riesgo'].value_counts())

## 4. Feature engineering

In [ ]:
df = engineer_features(df)
print(f'Columnas totales: {df.shape[1]}')
print('Nuevas features:', [c for c in df.columns if c not in load_ageb_v3().columns])

## 5. Guardar dataset procesado

In [ ]:
from src.config import AGBS_REGEN_DIR
path = save_path(AGBS_REGEN_DIR, 'ageb_v3_procesadas.csv')
df.to_csv(path, index=False)
print(f'Guardado en: {path}')

## 6. Validación final

In [ ]:
validate_dataframe(df, expected_rows=EXPECTED_ROWS,
                   required_cols=FEATURE_COLS_RAW + [TARGET_COL, 'alto_riesgo'],
                   name='Dataset final')
print('✅ Limpieza completada exitosamente.')